<a href="https://colab.research.google.com/github/f0rceFOS/python-practice-algos/blob/master/636-01/hw_2_vinogradov_a_a.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Задача «Банкомат»

В банкомате есть купюры следующих номиналов:
100, 200, 500, 1000, 2000, 5000 рублей.

Количество купюр каждого номинала ограничено и задано в массиве limits (len(limits) = 6), где limits[i] — количество купюр номинала denominations[i].
Номиналы упорядочены по возрастанию:
denominations = [100, 200, 500, 1000, 2000, 5000].

Клиент хочет снять некоторую сумму amount. Банкомат должен выдать эту сумму минимальным общим количеством купюр.

Если выдать сумму невозможно, вернуть -1.

Если возможно, вернуть массив из 6 целых чисел, где result[i] — сколько купюр номинала denominations[i] нужно выдать.

Если существует несколько способов с одинаковым минимальным количеством купюр, можно вернуть любой из них.

#### Пример 1
Вход: amount = 700, limits = [10, 10, 10, 10, 10, 10]
Выход: [0, 1, 1, 0, 0, 0]
Пояснение: 700 = 200 + 500. Всего 2 купюры.
Другие варианты: 100 * 7 (7 купюр), 100 * 5 + 200 (6 купюр) и т.д.
Минимум 2 купюры.

#### Пример 2
Вход: amount = 300, limits = [0, 0, 0, 0, 0, 0]
Выход: -1
Пояснение: в банкомате нет купюр, выдать нельзя.

#### Пример 3
Вход: amount = 350, limits = [10, 0, 10, 10, 10, 10]
Выход: -1
Пояснение: Номиналы: 100, 500, 1000, 2000, 5000.
350 нельзя составить, так как нет купюр по 50.
Значит, -1.

#### Пример 4
Вход: amount = 0, limits = [0, 0, 0, 0, 0, 0]
Выход: [0, 0, 0, 0, 0, 0]
Пояснение: Сумма 0 всегда выдаётся 0 купюр.

#### Ограничения
0 <= amount <= 10^9
0 <= limits[i] <= 10^6
denominations = [100, 200, 500, 1000, 2000, 5000]

In [ ]:
def atm(amount, limits):
    """
    :type amount: int
    :type limits: List[int]
    :rtype: List[int] или int
    """
    denominations = [100, 200, 500, 1000, 2000, 5000]

    if amount == 0:
        return [0, 0, 0, 0, 0, 0]

    if amount < 0 or amount % 100 != 0:
        return -1

    if sum(denominations[i] * limits[i] for i in range(6)) < amount:
        return -1

    best = None
    best_count = float("inf")
    current = [0] * 6

    prefix_sum = [0] * 6
    total = 0
    for i in range(6):
        total += denominations[i] * limits[i]
        prefix_sum[i] = total

    def search(index, remaining, used_count):
        nonlocal best, best_count

        if remaining == 0:
            if used_count < best_count:
                best_count = used_count
                best = current.copy()
            return

        if index < 0 or used_count >= best_count:
            return

        if remaining > prefix_sum[index]:
            return

        denomination = denominations[index]
        max_quantity = min(limits[index], remaining // denomination)

        for quantity in range(max_quantity, -1, -1):
            new_remaining = remaining - quantity * denomination
            new_count = used_count + quantity

            if new_count >= best_count:
                continue

            if index > 0 and new_remaining > prefix_sum[index - 1]:
                continue

            current[index] = quantity
            search(index - 1, new_remaining, new_count)
            current[index] = 0

    search(5, amount, 0)

    return best if best is not None else -1


In [ ]:
print(atm(700, [10, 10, 10, 10, 10, 10]))
print(atm(300, [0, 0, 0, 0, 0, 0]))
print(atm(350, [10, 10, 10, 10, 10, 10]))
print(atm(0, [0, 0, 0, 0, 0, 0]))

print(atm(600, [0, 3, 1, 0, 0, 0]))


[0, 1, 1, 0, 0, 0]
-1
-1
[0, 0, 0, 0, 0, 0]
[0, 3, 0, 0, 0, 0]


Время: в худшем случае перебор вариантов количества купюр (с отсечениями).  
Память: O(1), так как число номиналов фиксировано (6).

### 1. Задача «Подарочная карта»
У вас есть подарочная карта магазина с балансом target рублей.
Вы пришли в магазин, где каждый товар имеет цену prices[i].
Вы хотите купить ровно два товара так, чтобы их суммарная стоимость точно равнялась балансу карты (потратить всё до копейки).

Верните индексы двух таких товаров (нумерация с 0).
Гарантируется, что ровно одно решение существует, и один и тот же товар нельзя купить дважды.

#### Пример 1
Вход: prices = [2, 7, 11, 15], target = 9
Выход: [0, 1]
Пояснение: prices[0] + prices[1] = 2 + 7 = 9.

#### Пример 2
Вход: prices = [3, 3], target = 6
Выход: [0, 1]
Пояснение: prices[0] + prices[1] = 3 + 3 = 6.

#### Ограничения
2 <= len(prices) <= 10^4
-10^9 <= prices[i] <= 10^9
-10^9 <= target <= 10^9
Ровно одно решение существует.

In [ ]:
def giftCard(prices, target):
    """
    :type prices: List[int]
    :type target: int
    :rtype: List[int] или int
    """
    seen = {}

    for i, price in enumerate(prices):
        need = target - price

        if need in seen:
            return [seen[need], i]

        seen[price] = i

    return -1


In [ ]:
print(giftCard([1, 3, 3, 9], 10))
print(giftCard([2, 7, 11, 15], 9))
print(giftCard([3, 3], 6))
print(giftCard([0, 0], 0))
print(giftCard([5, 3], 6))
print(giftCard([], 6))

[0, 3]
[0, 1]
[0, 1]
[0, 1]
-1
-1
